<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/07%20-%20Validade%20e%20Inferencia%20Logica%20na%20Seguranca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 07 - Notebook: Validade de Argumentos e Inferência Lógica na Segurança de Processos

Neste notebook implementamos a classe **`ProvadorDedutivoFormal`** para testar rigorosamente a validade lógica dos argumentos de segurança operacional da **Estação de Reabastecimento de Hidrogênio**.

Substituindo a antiga verificação manual baseada em Forward Chaining, exploramos agora métodos de prova analítica exaustiva por tabela-verdade (com $2^n$ estados), verificação algébrica por refutação (*Reductio ad Absurdum* / Modelos SAT) e a detecção algorítmica de falácias formais nas lógicas de intertravamento crítico.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Any

class ProvadorDedutivoFormal:
    @staticmethod
    def verificar_argumento_tabela_verdade(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Verifica a validade do argumento: P1, P2, ..., Pk |- C
        Um argumento é válido se e somente se, em TODA linha onde todas as premissas são TRUE,
        a conclusão também é estritamente TRUE.
        """
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_criticas = 0 
        linhas_validas = 0  
        contraexemplos = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            # Avalia a conjunção de todas as premissas para a valoração atual
            premissas_satisfeitas = all(p(env) for p in premissas)
            
            if premissas_satisfeitas:
                linhas_criticas += 1
                if conclusao(env):
                    linhas_validas += 1
                else:
                    contraexemplos.append(env)
                    
        valido = (linhas_criticas > 0) and (linhas_criticas == linhas_validas)
        
        return {
            "Total Estados (2^n)": total_estados,
            "Estados com Premissas True": linhas_criticas,
            "Estados com Conclusão True": linhas_validas,
            "Válido": valido,
            "Resultado Semântico": "ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA)" if valido else "FALÁCIA / ARGUMENTO INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Prova por Contradição / Refutação (Abordagem SAT Solver):
        O argumento P1..Pk |- C é válido se e somente se o conjunto {P1, ..., Pk, NOT C}
        for INSATISFATÍVEL (gerar uma contradição lógica).
        """
        n = len(variaveis)
        modelos_refutacao = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            # Procura um estado onde as premissas são verdadeiras, MAS a conclusão é falsa
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_refutacao.append(env)
                
        is_insatisfativel = len(modelos_refutacao) == 0
        return {
            "Satisfaz Negação": len(modelos_refutacao) > 0,
            "Refutação Bem-Sucedida": is_insatisfativel,
            "Conclusão": "PROVA POR CONTRADIÇÃO: ARGUMENTO VÁLIDO" if is_insatisfativel else "REFUTAÇÃO FALHOU: CONTRADIÇÃO NÃO ENCONTRADA"
        }

print("[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!")

[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!


In [2]:
# ==============================================================================
# BATERIA DE TESTES INDUSTRIAIS (ESTAÇÃO DE H2 - SETOR 100)
# ==============================================================================

# 1. Modus Ponens: (pt101_alta -> trip), pt101_alta |- trip
vars_mp = ['pt101_alta', 'trip']
p1_mp = lambda env: (not env['pt101_alta']) or env['trip']  # Regra de intertravamento configurada no CLP
p2_mp = lambda env: env['pt101_alta']                       # Fato: Sensor PT-101 detectou sobrepressão
c_mp  = lambda env: env['trip']                             # Conclusão deduzida: Trip é acionado

res_mp = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mp, [p1_mp, p2_mp], c_mp)
ref_mp = ProvadorDedutivoFormal.verificar_por_refutacao(vars_mp, [p1_mp, p2_mp], c_mp)

# 2. Modus Tollens: (trip -> fecha_xv101), not fecha_xv101 |- not trip
vars_mt = ['trip', 'fecha_xv101']
p1_mt = lambda env: (not env['trip']) or env['fecha_xv101'] # Regra: Trip de emergência isola a válvula
p2_mt = lambda env: not env['fecha_xv101']                  # Fato: A válvula XV-101 está aberta (operando)
c_mt  = lambda env: not env['trip']                         # Conclusão deduzida: O SIS não está em estado de Trip

res_mt = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mt, [p1_mt, p2_mt], c_mt)

# 3. Silogismo Hipotético: (vazamento_h2 -> fecha_xv101), (fecha_xv101 -> isola_s100) |- (vazamento_h2 -> isola_s100)
vars_sh = ['vazamento_h2', 'fecha_xv101', 'isola_s100']
p1_sh = lambda env: (not env['vazamento_h2']) or env['fecha_xv101']
p2_sh = lambda env: (not env['fecha_xv101']) or env['isola_s100']
c_sh  = lambda env: (not env['vazamento_h2']) or env['isola_s100']

res_sh = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sh, [p1_sh, p2_sh], c_sh)

# 4. Resolução Proposicional: (sobrepressao ou alta_temp), (not sobrepressao ou trip) |- (alta_temp ou trip)
vars_res = ['sobrepressao', 'alta_temp', 'trip']
p1_res = lambda env: env['sobrepressao'] or env['alta_temp']
p2_res = lambda env: (not env['sobrepressao']) or env['trip']
c_res  = lambda env: env['alta_temp'] or env['trip']

res_res = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_res, [p1_res, p2_res], c_res)

# 5. Falácia da Afirmação do Consequente (INVÁLIDO): (pt101_alta -> trip), trip |- pt101_alta
vars_fal = ['pt101_alta', 'trip']
p1_fal = lambda env: (not env['pt101_alta']) or env['trip'] # Regra real: sobrepressão causa trip
p2_fal = lambda env: env['trip']                            # O sistema entrou em Trip
c_fal  = lambda env: env['pt101_alta']                      # Falsa conclusão: O motivo exclusivo foi a pressão (Ignora temperatura ou gás)

res_fal = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal, [p1_fal, p2_fal], c_fal)

relatorio_testes = [
    {"Esquema Lógico": "Modus Ponens (MP)", "Variáveis": "pt101_alta, trip", "Resultado Semântico": res_mp["Resultado Semântico"], "Válido": res_mp["Válido"]},
    {"Esquema Lógico": "Modus Tollens (MT)", "Variáveis": "trip, fecha_xv101", "Resultado Semântico": res_mt["Resultado Semântico"], "Válido": res_mt["Válido"]},
    {"Esquema Lógico": "Silogismo Hipotético (SH)", "Variáveis": "vazamento_h2, fecha_xv101, isola_s100", "Resultado Semântico": res_sh["Resultado Semântico"], "Válido": res_sh["Válido"]},
    {"Esquema Lógico": "Resolução Proposicional (RES)", "Variáveis": "sobrepressao, alta_temp, trip", "Resultado Semântico": res_res["Resultado Semântico"], "Válido": res_res["Válido"]},
    {"Esquema Lógico": "Afirmação Consequente (Falácia)", "Variáveis": "pt101_alta, trip", "Resultado Semântico": res_fal["Resultado Semântico"], "Válido": res_fal["Válido"]}
]

print("=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS (ESTAÇÃO DE H2) ===")
print(formatar_tabela(relatorio_testes))

assert res_mp["Válido"] is True
assert res_mt["Válido"] is True
assert res_sh["Válido"] is True
assert res_res["Válido"] is True
assert res_fal["Válido"] is False
print("\n[OK] Todos os testes de inferência dedutiva e detecção de falácias do Setor 100 passaram com 100% de sucesso!\n")

=== RELATÓRIO DE VERIFICAÇÃO FORMAL DE ARGUMENTOS (ESTAÇÃO DE H2) ===
Esquema Lógico                  | Variáveis               | Resultado Semântico                     | Válido
--------------------------------+-------------------------+-----------------------------------------+-------
Modus Ponens (MP)               | pt101_alta, trip        | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Modus Tollens (MT)              | trip, fecha_xv101       | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Silogismo Hipotético (SH)       | vazamento, trip, alarme | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Resolução Proposicional (RES)   | sobrep, alta_temp, trip | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Afirmação Consequente (Falácia) | pt101_alta, trip        | FALÁCIA / ARGUMENTO INVÁLIDO            | False 

[OK] Todos os testes de inferência dedutiva e detecção de falácias do Setor 100 passaram com 100% de sucesso!
